In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
file_path = "Sales_Dataset.xlsx"

df = pd.read_excel(file_path, sheet_name="Sales Data")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

df["MarketingSpend"] = df["MarketingSpend"].fillna(
    df["MarketingSpend"].median()
)

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["MonthName"] = df["Date"].dt.strftime("%b")
df["Quarter"] = "Q" + df["Date"].dt.quarter.astype(str)
df["DayOfWeek"] = df["Date"].dt.day_name()

df.head()

In [ ]:
df.isnull().sum()

In [ ]:
total_sales = df["Sales"].sum()
total_orders = df["OrderID"].nunique()
total_units = df["UnitsSold"].sum()
average_sales = df["Sales"].mean()
median_sales = df["Sales"].median()
average_unit_price = df["UnitPrice"].mean()
average_discount = df["DiscountPct"].mean()
average_marketing_spend = df["MarketingSpend"].mean()

pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Orders",
        "Total Units Sold",
        "Average Order Sales",
        "Median Order Sales",
        "Average Unit Price",
        "Average Discount %",
        "Average Marketing Spend"
    ],
    "Value": [
        total_sales,
        total_orders,
        total_units,
        average_sales,
        median_sales,
        average_unit_price,
        average_discount,
        average_marketing_spend
    ]
})

In [ ]:
region_sales = (
    df.groupby("Region")
      .agg(
          Sales=("Sales", "sum"),
          Orders=("OrderID", "nunique"),
          UnitsSold=("UnitsSold", "sum"),
          AverageSales=("Sales", "mean")
      )
      .sort_values("Sales", ascending=False)
)

region_sales

In [ ]:
category_sales = (
    df.groupby("Category")
      .agg(
          Sales=("Sales", "sum"),
          Orders=("OrderID", "nunique"),
          UnitsSold=("UnitsSold", "sum"),
          AverageSales=("Sales", "mean")
      )
      .sort_values("Sales", ascending=False)
)

category_sales

In [ ]:
product_sales = (
    df.groupby("Product")
      .agg(
          Sales=("Sales", "sum"),
          Orders=("OrderID", "nunique"),
          UnitsSold=("UnitsSold", "sum"),
          AverageSales=("Sales", "mean")
      )
      .sort_values("Sales", ascending=False)
)

product_sales.head(10)

In [ ]:
segment_sales = (
    df.groupby("CustomerSegment")
      .agg(
          Sales=("Sales", "sum"),
          Orders=("OrderID", "nunique"),
          UnitsSold=("UnitsSold", "sum"),
          AverageSales=("Sales", "mean")
      )
      .sort_values("Sales", ascending=False)
)

segment_sales

In [ ]:
channel_sales = (
    df.groupby("Channel")
      .agg(
          Sales=("Sales", "sum"),
          Orders=("OrderID", "nunique"),
          UnitsSold=("UnitsSold", "sum"),
          AverageSales=("Sales", "mean")
      )
      .sort_values("Sales", ascending=False)
)

channel_sales

In [ ]:
monthly_sales = (
    df.groupby(df["Date"].dt.to_period("M"))
      .agg(
          Sales=("Sales", "sum"),
          Orders=("OrderID", "nunique"),
          UnitsSold=("UnitsSold", "sum")
      )
      .reset_index()
)

monthly_sales["Date"] = monthly_sales["Date"].astype(str)

monthly_sales

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales["Date"], monthly_sales["Sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(region_sales.index, region_sales["Sales"])
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(category_sales.index, category_sales["Sales"])
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
top_products = product_sales.head(10).sort_values("Sales")

plt.figure(figsize=(10, 6))
plt.barh(top_products.index, top_products["Sales"])
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(channel_sales.index, channel_sales["Sales"])
plt.title("Sales by Channel")
plt.xlabel("Channel")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = [
    "UnitsSold",
    "UnitPrice",
    "DiscountPct",
    "MarketingSpend",
    "Sales"
]

correlation = df[numeric_columns].corr()["Sales"].sort_values(ascending=False)
correlation

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["MarketingSpend"], df["Sales"], alpha=0.35)
plt.title("Marketing Spend vs Sales")
plt.xlabel("Marketing Spend")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
features = [
    "Region",
    "Category",
    "Product",
    "CustomerSegment",
    "Channel",
    "UnitsSold",
    "UnitPrice",
    "DiscountPct",
    "MarketingSpend",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeek"
]

target = "Sales"

X = df[features]
y = df[target]

categorical_features = [
    "Region",
    "Category",
    "Product",
    "CustomerSegment",
    "Channel",
    "Quarter",
    "DayOfWeek"
]

numerical_features = [
    "UnitsSold",
    "UnitPrice",
    "DiscountPct",
    "MarketingSpend",
    "Year",
    "Month"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        )
    ]
)

In [ ]:
regression_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            )
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

regression_model.fit(X_train, y_train)

In [ ]:
y_pred = regression_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

model_metrics = pd.DataFrame({
    "Metric": ["R²", "MAE", "RMSE"],
    "Value": [r2, mae, rmse]
})

model_metrics

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.35)

minimum = min(y_test.min(), y_pred.min())
maximum = max(y_test.max(), y_pred.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.title("Actual vs Predicted Sales")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.title("Residual Analysis")
plt.xlabel("Predicted Sales")
plt.ylabel("Residuals")
plt.tight_layout()
plt.show()

In [ ]:
df["SalesPerUnit"] = df["Sales"] / df["UnitsSold"]
df["ListValue"] = df["UnitsSold"] * df["UnitPrice"]
df["DiscountAmount"] = df["ListValue"] * df["DiscountPct"]

df.head()

In [ ]:
powerbi_data = df.copy()

powerbi_data.to_csv(
    "PowerBI_Ready_Sales.csv",
    index=False
)

powerbi_data.head()